# Agentic Systems: Detailed Overview

This notebook explains agentic systems and orchestration in a practical, implementation-oriented way.

It addresses:

- Tool and function calling
- Structured outputs
- Agent architectures
- Context window management
- Memory design
- Token optimization strategies
- Hallucination prevention (often misspelled as "hullaciagation")

The emphasis is on how to design robust systems, not only how to prompt models.

## 1) What is an agentic system?

A basic LLM application often does one call: prompt in, response out.

An **agentic system** adds control logic around the model so it can:

1. decide what to do next,
2. call tools,
3. inspect results,
4. revise plans,
5. stop when success criteria are met.

In other words, the model becomes one component in a larger decision loop.

### Why orchestration matters

Orchestration is the policy layer that decides:

- when to call the model,
- when to call tools,
- what state to persist,
- how to recover from failures,
- when to terminate.

Without orchestration, agents become expensive, hard to debug, and brittle.

In [ ]:
from __future__ import annotations

from dataclasses import dataclass, field
from typing import Any


@dataclass
class AgentState:
    user_goal: str
    plan: list[str] = field(default_factory=list)
    tool_results: list[dict[str, Any]] = field(default_factory=list)
    done: bool = False


def orchestrate_step(state: AgentState) -> str:
    """A minimal deterministic orchestrator for demonstration."""
    if not state.plan:
        state.plan = [
            "gather evidence",
            "synthesize answer",
            "validate grounding",
        ]

    next_action = state.plan.pop(0)

    if next_action == "validate grounding":
        state.done = True

    return next_action


state = AgentState(user_goal="Summarize Q2 business risks")

while not state.done:
    action = orchestrate_step(state)
    print(f"Orchestrator selected action: {action}")

## 2) Tool and function calling

Tool calling lets the agent interact with external capabilities such as search, databases, calculators, APIs, and internal business systems.

A robust pattern includes:

- explicit tool schemas (name, parameters, return shape)
- permission boundaries (what tools are allowed)
- retries/timeouts/error handling
- audit logs for each call

The model should choose tools, but orchestration should enforce safety and policy.

In [ ]:
from typing import Callable


def tool_search_docs(query: str) -> dict[str, Any]:
    return {
        "tool": "search_docs",
        "query": query,
        "results": ["Result A", "Result B"],
    }


def tool_calculate(expression: str) -> dict[str, Any]:
    # Demo only: never use eval in production tool execution.
    safe_results = {
        "2+2": 4,
        "10/2": 5,
    }
    return {
        "tool": "calculate",
        "expression": expression,
        "value": safe_results.get(expression, "unsupported"),
    }


TOOLS: dict[str, Callable[..., dict[str, Any]]] = {
    "search_docs": tool_search_docs,
    "calculate": tool_calculate,
}


def call_tool(name: str, **kwargs: Any) -> dict[str, Any]:
    if name not in TOOLS:
        raise ValueError(f"Unknown tool: {name}")
    return TOOLS[name](**kwargs)


print(call_tool("search_docs", query="latest guidance"))
print(call_tool("calculate", expression="2+2"))

## 3) Structured outputs

Structured outputs are crucial in orchestration because downstream logic needs predictable machine-readable fields.

Instead of free text such as "I think we should call search", return objects like:

- `action`: `call_tool` / `final_answer`
- `tool_name`: name of selected tool
- `arguments`: validated argument object
- `confidence`: optional score

This improves reliability, testability, and observability.

In [ ]:
import json


REQUIRED_KEYS = {"action", "tool_name", "arguments"}


def validate_agent_decision(raw_json: str) -> dict[str, Any]:
    parsed = json.loads(raw_json)

    missing = REQUIRED_KEYS - set(parsed.keys())
    if missing:
        raise ValueError(f"Missing keys: {sorted(missing)}")

    if parsed["action"] not in {"call_tool", "final_answer"}:
        raise ValueError("Invalid action")

    if not isinstance(parsed["arguments"], dict):
        raise ValueError("arguments must be an object")

    return parsed


decision_json = json.dumps(
    {
        "action": "call_tool",
        "tool_name": "search_docs",
        "arguments": {"query": "credit risk trends"},
    }
)

print(validate_agent_decision(decision_json))

## 4) Agent architectures

Common architecture patterns:

- Router architecture: one model routes requests to specialized tools/agents.
- Planner-executor architecture: planner decomposes tasks; executor performs steps.
- Reflection architecture: agent critiques and revises its own outputs.
- Multi-agent architecture: specialized agents collaborate with coordination logic.

Design tradeoff:

- More components increase capability and robustness.
- More components also increase latency, cost, and operational complexity.

## 5) Context window management

Context windows are limited. If you exceed the model window, information is truncated or calls fail.

Effective management strategies:

- retrieval filtering: include only relevant chunks
- summarization: compress history or documents
- sliding window: keep recent turns plus condensed older memory
- section prioritization: include high-value context first

In production, context construction should be deterministic and budget-aware.

In [ ]:
def build_context_with_budget(
    chunks: list[str],
    token_budget: int,
    separator_tokens: int = 3,
) -> tuple[list[str], int]:
    """Approximation: 1 token ~= 0.75 word is model-dependent; we use word count proxy."""
    selected: list[str] = []
    used = 0

    for chunk in chunks:
        chunk_tokens = len(chunk.split()) + separator_tokens
        if selected and used + chunk_tokens > token_budget:
            continue
        selected.append(chunk)
        used += chunk_tokens
        if used >= token_budget:
            break

    return selected, used


demo_chunks = [
    "Market conditions are volatile but improving in select regions.",
    "Credit spread dynamics indicate elevated short-term risk.",
    "Management guidance expects margin normalization in H2.",
]

selected_chunks, used_tokens = build_context_with_budget(demo_chunks, token_budget=20)
print(selected_chunks)
print("Estimated tokens used:", used_tokens)

## 6) Memory in agentic systems

Memory is usually split into layers:

- Working memory: current turn/task state.
- Episodic memory: past interactions and outcomes.
- Semantic memory: durable knowledge summaries/facts.
- External memory: vector stores, databases, files.

A good memory system prioritizes relevance and recency, not just volume.

Memory should be queryable, compressible, and governed by retention rules.

In [ ]:
from datetime import datetime


memory_events: list[dict[str, Any]] = [
    {
        "ts": "2026-07-10T10:00:00",
        "topic": "risk",
        "note": "User asked about short-term credit risk.",
    },
    {
        "ts": "2026-07-10T11:00:00",
        "topic": "revenue",
        "note": "User asked for revenue sensitivity drivers.",
    },
]


def retrieve_memory(topic: str, limit: int = 2) -> list[dict[str, Any]]:
    filtered = [m for m in memory_events if topic.lower() in m["topic"].lower()]
    filtered.sort(key=lambda x: datetime.fromisoformat(x["ts"]), reverse=True)
    return filtered[:limit]


print(retrieve_memory("risk"))

## 7) Token optimization strategy

High-token prompts are expensive and increase latency. Token optimization should be built into orchestration.

Practical strategy:

1. classify intent first (route to smallest capable path)
2. retrieve broadly for recall
3. rerank aggressively for precision
4. pass only top evidence under a token budget
5. keep reusable summaries in memory/index
6. enforce output length constraints

This pattern usually preserves answer quality while lowering cost.

## 8) How to avoid hallucination

Hallucination is confident but unsupported generation.

Mitigation stack (best used together):

- Grounding: require evidence from trusted sources or tools.
- Citation-first prompting: ask for source references per claim.
- Verification pass: run a second check stage for factual consistency.
- Abstention policy: allow "I do not know" when evidence is missing.
- Constraint prompts: restrict answer to retrieved/tool-provided context.
- Schema validation: reject malformed outputs before execution.

For high-stakes domains, pair agent answers with deterministic checks and human review.

In [ ]:
def grounded_answer_policy(answer: str, evidence: list[str]) -> dict[str, Any]:
    """Simple heuristic policy: flag if answer has claims not traceable to evidence terms."""
    answer_terms = normalize_tokens(answer)
    evidence_terms = set()
    for piece in evidence:
        evidence_terms |= normalize_tokens(piece)

    unsupported = sorted(term for term in answer_terms if term not in evidence_terms)
    supported_ratio = 1.0 - (len(unsupported) / max(1, len(answer_terms)))

    return {
        "supported_ratio": round(supported_ratio, 3),
        "unsupported_terms_sample": unsupported[:12],
        "status": "pass" if supported_ratio >= 0.75 else "review",
    }


evidence_snippets = [
    "NVIDIA reported strong data center demand and revenue growth.",
    "Guidance emphasized AI infrastructure expansion.",
]
candidate_answer = "NVIDIA reported strong data center demand and opened a new retail banking segment."

print(grounded_answer_policy(candidate_answer, evidence_snippets))

## 9) Final checklist for production agentic systems

Before shipping, verify:

- clear orchestration state machine
- strict tool schema and permissions
- structured outputs with validation
- context budget enforcement
- memory retention and retrieval policy
- token and latency telemetry
- hallucination checks and abstention behavior
- test scenarios for failures and retries

A reliable agentic system is mostly systems engineering around the model, not just prompting.